# Notebook 03 · What the carbon map cannot see: a satellite time series

**Exercise:** Carbon storage change in mainland Portugal, 1990–2018
**Companion to:** Pedro Cabral, *Monitoring Dynamics in Ecosystem Service Indicators with Remote Sensing* (Elsevier)

Notebook 02 produced a carbon indicator from two land-cover snapshots. It cannot see anything that happens *within* a land-cover class, and it cannot see time between the snapshots. This notebook brings in what the book is about: a continuous satellite record of vegetation, here 25 years of MODIS NDVI, used to test what the snapshot indicator misses.

The default study area is central Portugal, where the wildfires of June and October 2017 (Pedrógão Grande, Lousã, Oliveira do Hospital and surroundings) burned large areas of forest and shrubland. Set `REGION = "mainland"` to run the whole country; it takes considerably longer.

| Step | What you do | Book section |
|---|---|---|
| 1 | Choose and load a long, consistent record | 3.2, 4.2.2 |
| 2 | Mask poor-quality observations and build annual composites | 3.3 |
| 3 | Compute per-pixel trends | 5.3.1 |
| 4 | Find change hidden inside unchanged land-cover classes | 4.7.1, 5.3.5 |
| 5 | Follow the 2017 fires: disturbance and recovery | 5.3.4, 5.4 |
| 6 | Compare burned and matched unburned pixels | 5.7.1 |
| 7 | Confront the result with Notebook 02 | 4.6.4, 4.7.1 |
| 8 | Report a land-productivity trajectory | SDG indicator 15.3.1 |

**Data access.** NDVI is read from the [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/dataset/modis-13Q1-061) STAC catalogue, which needs no account. The first run downloads and caches the annual composites in `outputs/`; later runs read the cache. Run Notebook 01 first.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject

import es_utils as es

DATA, OUT = es.DATA_DIR, es.OUT_DIR
OUT.mkdir(exist_ok=True)

REGIONS = {  # lon_min, lat_min, lon_max, lat_max
    "central_fires_2017": (-8.65, 39.60, -7.60, 40.45),
    "mainland": (-9.55, 36.95, -6.15, 42.20),
}
REGION = "central_fires_2017"
START_YEAR, END_YEAR, FIRE_YEAR = 2001, 2025, 2017
RES_M = 250
warnings.filterwarnings("ignore", message="Mean of empty slice")
PIX_KHA = (RES_M / 100) ** 2 / 1e3  # area of one pixel in thousand hectares

## Step 1 · Choosing the record

> **Book link · Section 3.2 (data sources) and Section 4.2.2 (temporal consistency often matters more than spatial detail).**

| Option | Resolution | Record | Why (not) here |
|---|---|---|---|
| MODIS MOD13Q1 NDVI | 250 m, 16-day | 2000–present, one sensor family | **Chosen**: long, internally consistent, light to process |
| Landsat 5/7/8/9 | 30 m, 16-day | 1984–present | Finer, back to the 1990 map, but sensor transitions need harmonization (Section 3.3) |
| Sentinel-2 | 10–20 m, 5-day | 2015–present | Too short to cover the pre-fire baseline and a trend |

The cost of the choice is visible later: at 250 m, one MODIS pixel covers 6.25 CLC pixels, so CLC must be aggregated to match.

The grid is defined once, in EPSG:3035 at 250 m, so the NDVI and the CLC maps from Notebook 01 line up exactly.

In [ ]:
from odc.geo.geobox import GeoBox
from pyproj import Transformer

bbox = REGIONS[REGION]
x0, y0, x1, y1 = Transformer.from_crs(4326, 3035, always_xy=True).transform_bounds(*bbox)
snap = lambda v, f: f(v / RES_M) * RES_M
geobox = GeoBox.from_bbox((snap(x0, np.floor), snap(y0, np.floor), snap(x1, np.ceil), snap(y1, np.ceil)),
                          crs="EPSG:3035", resolution=RES_M)
grid = {"crs": "EPSG:3035", "transform": rasterio.Affine(*geobox.affine[:6]),
        "height": geobox.shape[0], "width": geobox.shape[1]}
print(f"Target grid: {grid['height']} x {grid['width']} pixels at {RES_M} m")

## Step 2 · Quality masking and annual composites

> **Book link · Section 3.3 (preprocessing and temporal compositing).** Each 16-day MODIS composite carries a *pixel reliability* flag: 0 = good, 1 = marginal, 2 = snow or ice, 3 = cloudy, −1 = no data. Only 0 and 1 are kept. The annual composite is the median of the remaining observations, which is robust to residual cloud and to occasional outliers.

Reprojection from the MODIS sinusoidal grid (Section 3.5) uses nearest neighbour for both bands, so that every NDVI value keeps the quality flag that belongs to it.

In [ ]:
NDVI_BAND, QA_BAND = "250m_16_days_NDVI", "250m_16_days_pixel_reliability"
CACHE = OUT / f"ndvi_annual_{REGION}_{START_YEAR}_{END_YEAR}_{RES_M}m.npz"


def annual_ndvi(year, catalog):
    import odc.stac
    items = catalog.search(collections=["modis-13Q1-061"], bbox=bbox,
                           datetime=f"{year}-01-01/{year}-12-31").item_collection()
    if len(items) == 0:
        raise RuntimeError(f"No MODIS items found for {year}.")
    missing = {NDVI_BAND, QA_BAND} - set(items[0].assets)
    if missing:
        raise KeyError(f"Assets {missing} not found; available: {sorted(items[0].assets)}")
    # Load as float32: one band's declared nodata does not fit its integer type,
    # and float lets missing pixels become NaN instead.
    ds = odc.stac.load(items, bands=[NDVI_BAND, QA_BAND], geobox=geobox, dtype="float32",
                       resampling="nearest", groupby="solar_day", fail_on_error=False)
    ndvi = ds[NDVI_BAND]
    good = (ndvi >= -2000) & ds[QA_BAND].isin([0, 1])
    composite = (ndvi.where(good) * 1e-4).median("time", skipna=True)
    return composite.values.astype("float32"), int(good.sum("time").median())


if CACHE.exists():
    z = np.load(CACHE)
    cube, years = z["cube"], z["years"]
    print(f"Loaded cached composites {years[0]}–{years[-1]} from {CACHE.name}")
else:
    import planetary_computer
    import pystac_client
    catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                                        modifier=planetary_computer.sign_inplace)
    years = np.arange(START_YEAR, END_YEAR + 1)
    layers = []
    for y in years:
        comp, n_obs = annual_ndvi(int(y), catalog)
        layers.append(comp)
        print(f"{y}: median of {n_obs} reliable observations per pixel")
    cube = np.stack(layers)
    np.savez_compressed(CACHE, cube=cube, years=years)
print("NDVI cube (years, rows, cols):", cube.shape)

In [ ]:
show = [FIRE_YEAR - 1, FIRE_YEAR + 1, int(years[-1])]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, y in zip(axes, show):
    es.plot_continuous(ax, cube[list(years).index(y)], grid, f"Annual median NDVI, {y}", "NDVI",
                       cmap="YlGn", vmin=0, vmax=0.9)
plt.tight_layout()

## Step 3 · Per-pixel trends

> **Book link · Section 5.3.1 (degradation signals in long-term trajectories).** A trajectory says more than two snapshots: it shows direction, rate and whether a change persisted.

For each pixel, the **Theil–Sen slope** gives a robust rate of change (NDVI per year), and the **Mann–Kendall** statistic tests whether the monotonic trend is distinguishable from noise (|Z| > 1.96, roughly 95%). Neither assumes a linear or normally distributed signal. Note that a fire followed by recovery is not monotonic; the trend summarizes it, and Step 5 looks at it directly.

In [ ]:
slope, zstat = es.theil_sen_mann_kendall(cube, years)
TREND = np.full(slope.shape, -1, np.int8)
TREND[~np.isnan(zstat)] = 1
TREND[zstat < -1.96] = 0
TREND[zstat > 1.96] = 2
TREND_NAMES = ["Declining", "Stable", "Improving"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
es.plot_continuous(axes[0], slope * 10, grid, f"Theil–Sen NDVI slope, {years[0]}–{years[-1]}",
                   "NDVI change per decade", cmap="BrBG", diverging=True, vmax=0.15)
from matplotlib.colors import ListedColormap
trend_cmap = ListedColormap(["#d73027", "#f0f0f0", "#1a9850"])
axes[1].imshow(np.ma.masked_less(TREND, 0), cmap=trend_cmap, vmin=-0.5, vmax=2.5,
               extent=es.extent(grid), interpolation="nearest")
axes[1].set_title("Mann–Kendall trend (|Z| > 1.96)")
axes[1].set_xticks([]); axes[1].set_yticks([])
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color=c, label=n) for c, n in zip(trend_cmap.colors, TREND_NAMES)],
               loc="lower left", fontsize=8)
plt.tight_layout()
fig.savefig(OUT / f"03_ndvi_trend_{REGION}.png", dpi=150)

## Step 4 · Change hidden inside unchanged land cover

> **Book link · Section 4.7.1 (treating indicators as services) and Section 5.3.5 (the limits of land-cover proxies).** In Notebook 02, a pixel whose CLC class did not change contributes exactly zero carbon change. Here we ask how often the satellite record disagrees.

CLC is first aggregated from 100 m to the 250 m grid with the **mode** (majority class), the categorical counterpart of averaging (Section 3.3).

In [ ]:
def to_grid(path):
    codes, prof = es.read_clc(path)
    out = np.full((grid["height"], grid["width"]), -1, np.int16)
    reproject(codes, out, src_transform=prof["transform"], src_crs=prof["crs"],
              dst_transform=grid["transform"], dst_crs=grid["crs"],
              src_nodata=-1, dst_nodata=-1, resampling=Resampling.mode)
    return out

c90, c18 = to_grid(DATA / "clc1990_portugal.tif"), to_grid(DATA / "clc2018_portugal.tif")
g90, g18 = es.to_groups(c90), es.to_groups(c18)
land = (g90 >= 0) & (g18 >= 0) & ~np.isin(g18, [es.GROUP_ORDER.index("Water"), es.GROUP_ORDER.index("Wetlands")]) & (TREND >= 0)
same = land & (c90 == c18)

rows = []
for gi, name in enumerate(es.GROUP_ORDER):
    sel = same & (g18 == gi)
    if sel.sum() < 50:
        continue
    shares = [100 * np.mean(TREND[sel] == k) for k in range(3)]
    rows.append([name, sel.sum() * PIX_KHA] + shares)
hidden = pd.DataFrame(rows, columns=["land cover (unchanged 1990–2018)", "area (kha)"] +
                      [f"{n} (%)" for n in TREND_NAMES])
display(hidden.round(1))
print(f"Unchanged land cover with a significant NDVI trend: "
      f"{100 * np.mean(TREND[same] != 1):.1f}% of {same.sum() * PIX_KHA:,.0f} kha")

In [ ]:
hidden_map = np.where(same, TREND, -1).astype(np.int8)
hidden_map[land & ~same] = 3
cmap = ListedColormap(["#d73027", "#f0f0f0", "#1a9850", "#bdbdbd"])
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(np.ma.masked_less(hidden_map, 0), cmap=cmap, vmin=-0.5, vmax=3.5,
          extent=es.extent(grid), interpolation="nearest")
ax.legend(handles=[Patch(color=c, label=l) for c, l in zip(cmap.colors,
          ["Same class, NDVI declining", "Same class, NDVI stable", "Same class, NDVI improving",
           "CLC class changed"])], loc="lower left", fontsize=8)
ax.set_title("Where the snapshot indicator reports zero change")
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
fig.savefig(OUT / f"03_hidden_change_{REGION}.png", dpi=150)

Every red or green pixel above has a carbon change of exactly zero in Notebook 02. Some of this is real within-class change (forest growth, thinning, die-back, drought, fires after 2018), some is noise or inter-annual variability that the trend test did not filter out, and some reflects the coarser MODIS grid. NDVI is itself a proxy for vegetation activity, not for carbon stock (Section 4.2.1), so this does not tell us how much carbon is missing. It tells us *where the land-cover indicator is blind*.

## Step 5 · The 2017 fires: disturbance and recovery

> **Book link · Section 5.3.4 (degradation versus disturbance) and Section 5.4 (recovery).** A disturbance is a sharp drop followed, possibly, by recovery. Degradation is a sustained decline. Two snapshots, 1990 and 2018, cannot tell them apart; a trajectory can.

Burned pixels are identified with a simple NDVI-difference proxy: a drop of more than 0.15 between the mean of the three pre-fire years and the year after the fire. Operational burned-area products such as MODIS MCD64A1 or the Portuguese ICNF fire perimeters are better references, and a good extension is to replace the proxy with one of them.

In [ ]:
yi = {int(y): i for i, y in enumerate(years)}
pre = np.nanmean(cube[[yi[FIRE_YEAR - k] for k in (1, 2, 3)]], axis=0)
post = cube[yi[FIRE_YEAR + 1]]
dndvi = post - pre
burned = land & (dndvi < -0.15) & (pre > 0.3)
unburned = land & (dndvi > -0.05) & (pre > 0.3)
print(f"Burned (NDVI proxy): {burned.sum() * PIX_KHA:,.0f} kha;  candidate controls: {unburned.sum() * PIX_KHA:,.0f} kha")

## Step 6 · Burned against comparable unburned pixels

> **Book link · Section 5.7.1 (before–after–control–impact).** The burned pixels' trajectory alone mixes the fire's effect with everything else that happened in those years, such as drought. Comparing it with unburned pixels that were similar before the fire removes the shared part.

Controls are matched on two pre-fire characteristics: land-cover group in 1990 and pre-fire NDVI (bins of 0.02). Each control is weighted so that, together, they reproduce the burned pixels' distribution.

In [ ]:
bins = np.floor(np.nan_to_num(pre, nan=-1.0) / 0.02).astype(np.int32)
key = np.where(land, g90.astype(np.int32) * 1000 + bins, -1)
kb, nb = np.unique(key[burned], return_counts=True)
kc, nc = np.unique(key[unburned], return_counts=True)
w_of = dict(zip(kc, np.zeros(len(kc))))
common = np.intersect1d(kb, kc)
for k in common:
    w_of[k] = nb[kb == k][0] / nc[kc == k][0]
weights = np.zeros(key.shape, np.float32)
ctrl = unburned & np.isin(key, common)
weights[ctrl] = np.vectorize(w_of.get)(key[ctrl])
treat = burned & np.isin(key, common)
print(f"Matched: {treat.sum():,} burned pixels with {ctrl.sum():,} weighted controls "
      f"({100 * treat.sum() / max(burned.sum(), 1):.0f}% of burned pixels have a match)")

traj_b = np.array([np.nanmean(cube[i][treat]) for i in range(len(years))])
traj_c = np.array([np.nansum(cube[i][ctrl] * weights[ctrl]) / weights[ctrl].sum() for i in range(len(years))])
before = years < FIRE_YEAR
did = (traj_b - traj_c) - np.mean((traj_b - traj_c)[before])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(years, traj_b, "o-", color="#d73027", label="Burned (proxy)")
axes[0].plot(years, traj_c, "o-", color="#4575b4", label="Matched unburned")
axes[1].bar(years, did, color=np.where(did < 0, "#d73027", "#1a9850"))
for ax in axes:
    ax.axvline(FIRE_YEAR, color="grey", ls="--")
axes[0].set_ylabel("Annual median NDVI"); axes[0].legend(); axes[0].set_title("Trajectories")
axes[1].set_ylabel("Δ NDVI relative to pre-fire gap")
axes[1].set_title("Difference-in-differences: fire effect by year")
plt.tight_layout()
fig.savefig(OUT / f"03_fire_did_{REGION}.png", dpi=150)

recovered_year = np.full(pre.shape, np.nan, np.float32)
for y in range(FIRE_YEAR + 1, int(years[-1]) + 1):
    hit = treat & np.isnan(recovered_year) & (cube[yi[y]] >= 0.9 * pre)
    recovered_year[hit] = y
share = 100 * np.mean(~np.isnan(recovered_year[treat]))
print(f"By {years[-1]}, {share:.0f}% of burned pixels had regained at least 90% of their pre-fire NDVI.")
display(pd.Series(recovered_year[treat]).value_counts().sort_index().rename("pixels").to_frame()
        .rename_axis("first year ≥ 90% of pre-fire NDVI"))

## Step 7 · Confronting the satellite record with Notebook 02

> **Book link · Section 4.6.4 (interpreting model outputs as indicators) and Section 4.7.1.** Notebook 02 assigned the burned pixels a large, permanent carbon loss, most of it soil carbon. What does the vegetation record say about the same pixels?

In [ ]:
table = pd.read_csv(DATA / "carbon_pools.csv")
table["c_total"] = table[es.POOLS].sum(axis=1)
d_total = es.lookup(c18, table, "c_total") - es.lookup(c90, table, "c_total")
d_soil = es.lookup(c18, table, "c_soil") - es.lookup(c90, table, "c_soil")
burnt_label = treat & (c18 == 33)

comparison = pd.Series({
    "Burned pixels (NDVI proxy), kha": treat.sum() * PIX_KHA,
    "…of which labelled 'Burnt areas' in CLC 2018 (%)": 100 * burnt_label.sum() / max(treat.sum(), 1),
    "Carbon change implied by CLC + table (t C/ha)": np.nanmean(d_total[treat]),
    "…of which soil carbon (t C/ha)": np.nanmean(d_soil[treat]),
    f"NDVI {years[-1]} as % of pre-fire NDVI": 100 * np.nanmean(cube[-1][treat] / pre[treat]),
    "Pixels back to ≥ 90% of pre-fire NDVI (%)": share,
})
display(comparison.round(1).to_frame("value"))

The two indicators answer different questions, and neither is the service. Still, the contrast is instructive: the land-cover indicator records the label at one moment and converts it into a permanent stock change, while the time series shows a disturbance followed by partial recovery. A defensible carbon assessment of these areas would report the fire as a disturbance, treat the soil-carbon loss as an artefact of the class value, and monitor the recovery trajectory instead of a single post-fire label.

## Step 8 · A land-productivity trajectory for SDG 15.3.1

> **Book link · Chapter 1 and Chapter 4 (SDG bridging).** SDG indicator 15.3.1, the proportion of land that is degraded over total land area, combines three sub-indicators: land cover, land productivity and soil organic carbon. Its land-productivity *trajectory* sub-indicator is essentially the per-pixel NDVI trend with a significance test computed in Step 3.

This is a simplified version: the official Good Practice Guidance (and tools such as Trends.Earth) define baseline and reporting periods, combine trajectory with state and performance metrics, and use a vegetation index integrated over the growing season.

In [ ]:
prod = pd.Series([np.sum(land & (TREND == k)) * PIX_KHA for k in range(3)], index=TREND_NAMES, name="area (kha)")
prod_pct = 100 * prod / prod.sum()
display(pd.DataFrame({"area (kha)": prod.round(0), "share (%)": prod_pct.round(1)}))
print(f"Share of land with a significantly declining productivity trajectory, {years[0]}–{years[-1]}: "
      f"{prod_pct['Declining']:.1f}%")

## Discussion questions

1. MODIS was chosen over Landsat for its consistency. Name one result in this notebook that would change at 30 m and one that would not. *(Sections 3.2, 4.2.2)*
2. Step 4 finds pixels with significant NDVI trends inside unchanged land-cover classes. List three processes in Portugal that could produce declining trends in unchanged forest, and say which could also lower carbon stocks. *(Sections 4.7.1, 5.3.5)*
3. The fire proxy is a 0.15 NDVI drop. How would a stricter threshold change the difference-in-differences result? Which kind of error (missed fires or false fires) matters more for the recovery estimate? *(Sections 5.3.4, 5.7.1)*
4. Controls were matched on 1990 land cover and pre-fire NDVI. What important difference between burned and unburned areas does this matching not capture? *(Section 5.7.1)*
5. Using Step 7, draft two sentences for a regional carbon report that describe the 2017 fires honestly, combining what both indicators show. *(Sections 4.6.4, 9.2)*
6. If Portugal reported the declining share from Step 8 under SDG 15.3.1, what would you add to the report to prevent post-fire recovery areas being misread as degradation? *(Section 5.3.4; SDG 15.3.1)*

---
*Data: MODIS MOD13Q1 v6.1 (Didan, K., NASA EOSDIS Land Processes DAAC), accessed through the Microsoft Planetary Computer. CORINE Land Cover, © European Union, Copernicus Land Monitoring Service, EEA.*